# Código Normalizar Datasets

In [1]:
import pandas as pd
import re
from distritos import agregar_por_distrito

INPUT_DIR  = "../data/processed/"
OUTPUT_DIR = "../data/clean/"

TERRITORY_COL = "Territórios"
year_pattern  = re.compile(r"^(\d{4})")

FICHEIROS = [
    "Dataset 0-19.csv",
    "Dataset 20-34.csv",
    "Dataset 35-64.csv",
    "Dataset 65+.csv",
    "Dataset Total.csv"
]

In [2]:
for ficheiro in FICHEIROS:
    print(f"\n[A processar] {ficheiro}")
    df = pd.read_csv(INPUT_DIR + ficheiro)

    # Agrupa colunas pelo ano e soma as sub-colunas
    grupos_por_ano: dict[str, list[str]] = {}
    for col in df.columns:
        if col == TERRITORY_COL:
            continue
        m = year_pattern.match(col)
        if m:
            grupos_por_ano.setdefault(m.group(1), []).append(col)

    resultado = pd.DataFrame({TERRITORY_COL: df[TERRITORY_COL]})
    for ano in sorted(grupos_por_ano):
        resultado[ano] = df[grupos_por_ano[ano]].sum(axis=1)

    # Substitui municípios por distritos e soma os valores
    resultado = agregar_por_distrito(resultado, coluna_territorio=TERRITORY_COL)

    # Despivota: anos passam a linhas
    resultado = resultado.melt(
        id_vars=TERRITORY_COL,
        var_name="Ano",
        value_name="Valor"
    )
    resultado["Ano"] = resultado["Ano"].astype(int)
    resultado = resultado.sort_values([TERRITORY_COL, "Ano"]).reset_index(drop=True)

    resultado.to_csv(OUTPUT_DIR + ficheiro, index=False)
    print(f"Guardado: {OUTPUT_DIR}{ficheiro}")

print("\nConcluído!")


[A processar] Dataset 0-19.csv
Guardado: ../data/clean/Dataset 0-19.csv

[A processar] Dataset 20-34.csv
Guardado: ../data/clean/Dataset 20-34.csv

[A processar] Dataset 35-64.csv
Guardado: ../data/clean/Dataset 35-64.csv

[A processar] Dataset 65+.csv
Guardado: ../data/clean/Dataset 65+.csv

[A processar] Dataset Total.csv
Guardado: ../data/clean/Dataset Total.csv

Concluído!
